In [ ]:
!pip install xarray-spatial -q

In [ ]:
import xarray as xr
from matplotlib import pyplot as plt
import matplotlib
import numpy as np
import rioxarray
import rasterio
import xrspatial
import pystac_client
import requests
import os
from dask.diagnostics import ProgressBar
from dask.distributed import Client
import earthaccess
assert(earthaccess.login(strategy="netrc").authenticated)

from const import COARSEN_FACTOR

In [ ]:
template_ds = xr.open_zarr("../data_working/treemap2016_hostba_hydro.zarr/")
bounds = template_ds.rio.transform_bounds(4326)

In [ ]:
bounds

In [ ]:
client = pystac_client.Client.open("https://cmr.earthdata.nasa.gov/stac/LPCLOUD")
items = client.search(
    collections=["ASTGTM_003"],
    bbox=bounds
).item_collection()
print(f"Found {len(items)} items!")

In [ ]:
item_keys = [
    f"s3://lp-prod-protected/ASTGTM.003/{item.id}_dem.tif"
    for item in items
]

In [ ]:
endpoint = "https://data.lpdaac.earthdatacloud.nasa.gov/s3credentials"
creds = requests.get(endpoint).json()

In [ ]:
'''
session = rasterio.session.AWSSession(
    aws_secret_access_key=creds["secretAccessKey"],
    aws_access_key_id=creds["accessKeyId"],
    aws_session_token=creds["sessionToken"]
)
rio_env = rasterio.Env(
    session=session,
    GDAL_DISABLE_READDIR_ON_OPEN='TRUE',
    GDAL_HTTP_COOKIEFILE=os.path.expanduser('~/cookies.txt'),
    GDAL_HTTP_COOKIEJAR=os.path.expanduser('~/cookies.txt')
)
rio_env.__enter__()
'''

In [ ]:
def gdal_env_setup() -> None:
    '''
    Set environment variables for GDAL to work over S3.
    '''
    os.environ["AWS_ACCESS_KEY_ID"] = creds["accessKeyId"]
    os.environ["AWS_SECRET_ACCESS_KEY"] = creds["secretAccessKey"]
    os.environ["AWS_SESSION_TOKEN"] = creds["sessionToken"]
    os.environ["GDAL_DISABLE_READDIR_ON_OPEN"] = "TRUE"
    os.environ["GDAL_HTTP_COOKIEFILE"] = os.path.expanduser("~/cookies.txt")
    os.environ["GDAL_HTTP_COOKIEJAR"] = os.path.expanduser("~/cookies.txt")


In [ ]:
# Set the environment variables and try to read something
gdal_env_setup()

mytile = rioxarray.open_rasterio(item_keys[0], chunks={}).squeeze().compute()

There's a one-pixel overlap between tiles, so it seems easier to use dask futures instead of map_blocks to process each tile.

In [ ]:
# Be careful of azimuth vs. cartesian angle semantics!!
def heat_load_index(dem: xr.DataArray, z_factor: int=30) -> xr.DataArray:
    '''
    Calculate a heat load index from Eq. 1 in McCune and Keon (2002)
    10.1111/j.1654-1103.2002.tb02087.x
    '''
    # Folded aspect from McCune and Keon (2002)
    aspect = xrspatial.aspect(dem) * np.pi / 180 # azimuth angle in rads
    aspect_fold = np.pi - np.abs(aspect - np.pi)
    # Get slope manually to account for z factor
    # This comes from xrspatial's hillshade implementation
    x, y = np.gradient(dem.data)
    x = x / z_factor
    y = y / z_factor
    slope = np.arctan(np.sqrt(x**2 + y**2))

    # Other things we need for Eq. 1
    aspect_sin = np.sin(aspect_fold)
    aspect_cos = np.cos(aspect_fold)
    slope_sin  = np.sin(slope)
    slope_cos  = np.cos(slope)
    latitude = xr.ones_like(dem) * dem.y * np.pi / 180
    
    heat = -1.467 +\
     1.582 * np.cos(latitude) * slope_cos +\
    -1.500 * aspect_cos * slope_sin * np.sin(latitude)+\
    -0.262 * np.sin(latitude) * slope_sin +\
     0.607 * aspect_sin * slope_sin
    
    return heat

In [ ]:
def process_tile(key: str) -> xr.DataArray:
    '''
    Read a tile, calculate heat load index, coarsen, return
    elevation and heat load in one data array.
    '''
    tile = rioxarray.open_rasterio(key).squeeze().assign_coords(var="elev")
    heat = heat_load_index(tile).assign_coords(var="heat")

    return xr.concat([tile, heat], "var")\
        .coarsen(x=COARSEN_FACTOR, y=COARSEN_FACTOR, boundary="trim")\
        .mean()\
        .transpose("y", "x", "var")

In [ ]:
%%time
out = process_tile(item_keys[0])

In [ ]:
client = Client()
# Enter rio environment on workers to allow reading
# from S3.
client.run(lambda: gdal_env_setup())
client

In [ ]:
futures = client.map(process_tile, item_keys)
results = client.gather(futures)

In [ ]:
topo_ds = xr.combine_by_coords(results)

In [ ]:
topo_proj = topo_ds\
    .transpose("var", "y", "x")\
    .rio.reproject_match(template_ds, resampling=rasterio.enums.Resampling.bilinear)

In [ ]:
from cartopy import crs as ccrs
from cartopy import feature as cfeature

source_proj = ccrs.AlbersEqualArea(
    central_latitude=23,
    central_longitude=-96,
    standard_parallels=(29.5, 45.5)
)
target_proj = ccrs.Mercator()

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(9, 4), subplot_kw=dict(projection=target_proj))
xmin, ymin, xmax, ymax = topo_proj.rio.transform_bounds(3857)

for ax, var in zip(axes.flat, ("heat", "elev")):
    topo_proj.sel(var=var).plot(ax=ax, transform=source_proj, add_labels=False, xlim=[xmin, xmax], ylim=[ymin, ymax])
    ax.set_title(var)
    ax.coastlines()
    ax.add_feature(cfeature.STATES)

plt.show()

In [ ]:
topo_proj.to_zarr("../data_working/topo.zarr")